# Satisfactory — Power Grid Calculator

In [1]:
# Auto-reload data.py (and any other local module) when it changes on disk,
# so editing data.py doesn't require restarting the kernel.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math;
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dataclasses import dataclass, field
from typing import Optional

import data  # all Satisfactory reference data + clock-speed helpers

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

In [2]:
# ── Choices — change these and everything below updates ──────────────────────
desired_power = 10_000            # MW target
generator     = data.FUEL_GEN     # generator to use
fuel          = 'turbofuel'       # fuel to burn — 'rocket_fuel', 'fuel', 'ionized_fuel' etc
clock         = generator['max_clock']   # run at max overclock (2.5 = 250%)

# ── Derived from data.py — no hard-coded numbers ─────────────────────────────
fuel_energy    = data.ENERGY[fuel]                     # MJ per unit
fuel_unit      = generator['fuels'][fuel]['unit']      # e.g. 'm3/min'
base_burn_rate = generator['fuels'][fuel]['rate']      # units/min @ 100% clock
burn_duration  = fuel_energy / generator['power_mw']   # seconds to burn 1 unit @ 100%

power_per_gen  = data.gen_output(generator['power_mw'], clock)    # MW per generator @ clock
num_generators = math.ceil(desired_power / power_per_gen)
burn_rate      = data.gen_consumption(base_burn_rate, clock)      # units/min per gen @ clock (note the ×clock!)
total_fuel     = num_generators * burn_rate                       # total fuel/min @ clock
total_power    = num_generators * power_per_gen                   # actual MW produced

In [3]:
# ── Present the plan as a styled table ───────────────────────────────────────
from IPython.display import display, Markdown

fuel_label = fuel.replace('_', ' ').title()

display(Markdown(f"### ⚡ Power Plan — {desired_power:,} MW from {fuel_label}"))

plan = [
    ("Generator",            generator['name']),
    ("Clock speed",          f"{clock:.0%}"),
    ("Power per generator",  f"{power_per_gen:,.0f} MW"),
    ("Generators needed",    f"{num_generators}"),
    ("Total power produced", f"{total_power:,.0f} MW"),
    ("Fuel",                 fuel_label),
    ("Energy per unit",      f"{fuel_energy:,} MJ"),
    ("Burn rate per gen",    f"{burn_rate:,.2f} {fuel_unit}"),
    ("Total fuel needed",    f"{total_fuel:,.1f} {fuel_unit}"),
]
summary = pd.DataFrame(plan, columns=["Metric", "Value"])

(summary.style
    .hide(axis="index")
    .set_caption(f"{num_generators}× {generator['name']} @ {clock:.0%}")
    .set_properties(subset=["Value"],  **{"text-align": "right", "font-weight": "bold"})
    .set_properties(subset=["Metric"], **{"text-align": "left"})
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-style", "italic"),
                                          ("padding-top", "8px"), ("opacity", "0.7")]},
        {"selector": "th",      "props": [("text-align", "left"), ("font-size", "1.05em")]},
        {"selector": "td, th",  "props": [("padding", "4px 14px")]},
    ]))

### ⚡ Power Plan — 10,000 MW from Turbofuel

Metric,Value
Generator,Fuel-Powered Generator
Clock speed,250%
Power per generator,625 MW
Generators needed,16
Total power produced,"10,000 MW"
Fuel,Turbofuel
Energy per unit,"2,000 MJ"
Burn rate per gen,18.75 m3/min
Total fuel needed,300.0 m3/min


---
## ⛓️ Resource Chain — back to raw

Trace the chosen fuel back to mined/extracted resources with `data.resolve_to_raw(fuel, total_fuel)` — standard recipes only. Change `fuel` in the setup cell and every table below re-derives automatically.

In [4]:
# Resolve the chosen fuel back to raw resources (standard recipes).
# Overclock every production building to max — this shrinks the machine COUNT only;
# raw-resource and byproduct rates are throughput totals, unchanged by overclocking.
production_clock = max(data.POWER_SHARD_CAPS.values())   # 2.5 = 250%

chain = data.resolve_to_raw(fuel, total_fuel, clock=production_clock)
tree  = data.chain_tree(fuel, total_fuel, clock=production_clock)

In [5]:
# ── Render the chain as a production tree ────────────────────────────────────
from IPython.display import display, HTML

def label(s):
    """'compacted_coal' → 'Compacted Coal'."""
    return s.replace('_', ' ').title()

# 1) Flatten the hierarchy into rows, drawing ├─ └─ connectors; tally buildings
rows = []
building_count = 0
def _walk(node, prefix="", is_last=True, is_root=True):
    global building_count
    name = label(node['item'])
    tree_label = name if is_root else prefix + ("└─ " if is_last else "├─ ") + name
    rate = f"{node['rate']:,.0f}/min"
    if node['raw']:
        info = "⛏ raw"
    else:
        machines = math.ceil(node['machines'])
        building_count += machines
        info = f"{machines}× {node['machine']}"
        if node['clock'] != 1.0:
            info += f" @ {node['clock']:.0%}"
        for bp, r in node['byproducts'].items():
            info += f"   ↳ +{r:,.0f} {label(bp)}"
    rows.append((tree_label, rate, info))
    kids = node['children']
    child_prefix = prefix + ("   " if is_last else "│  ")
    for i, k in enumerate(kids):
        _walk(k, child_prefix, i == len(kids) - 1, is_root=False)
_walk(tree)

# 2) Align the label and rate columns
lw = max(len(r[0]) for r in rows)
rw = max(len(r[1]) for r in rows)
lines = [f"{a:<{lw}}   {b:>{rw}}   {c}" for a, b, c in rows]

# 3) Totals from the flat resolver
raw_str = " · ".join(f"{label(k)} {v:,.0f}" for k, v in sorted(chain['raw'].items()))
bp_str  = " · ".join(f"{label(k)} {v:,.0f}/min" for k, v in chain['byproducts'].items())
rule = "─" * max(len(l) for l in lines + [raw_str, bp_str or ""])

# 4) Assemble the panel
header = (f"⚡ {desired_power:,} MW  ◀  {total_fuel:,.0f} {fuel_unit} {label(fuel)}\n"
          f"   {num_generators}× {generator['name']} @ {clock:.0%}")
panel_lines = [header, rule, *lines, rule, f"RAW/min     {raw_str}"]
if bp_str:
    panel_lines.append(f"BYPRODUCT   {bp_str}   ⚠ must sink")
if production_clock != 1.0:
    panel_lines.append(
        f"OVERCLOCK   {building_count} production buildings @ {production_clock:.0%}"
        f" → {building_count * 3} power shards")
panel = "\n".join(panel_lines)

display(HTML(
    '<pre style="font-family:ui-monospace,Menlo,Consolas,monospace; line-height:1.5; '
    'font-size:0.95em; padding:14px 18px; border-left:3px solid #ff5da2; border-radius:4px; '
    'background:rgba(127,127,127,0.10); white-space:pre; overflow-x:auto">'
    f"{panel}</pre>"
))

In [6]:
# 